# Day 31 — EDA: class balance, geometry, samples

Status: COMPLETE. Provenance: `hf-vision/chest-xray-pneumonia` (CC BY 4.0),
verified identical to the Kaggle layout by split counts. 3/3 tests pass.

In [1]:
import sys
sys.path.insert(0, "..")
import json
from pathlib import Path

s = json.loads(Path("../models/eda_summary.json").read_text())
tr, te = s["train"], s["test"]
print(f"train: {tr['n']} (PNEUMONIA {tr['counts']['PNEUMONIA']} / "
      f"{tr['counts']['PNEUMONIA']/tr['n']*100:.1f}%, NORMAL {tr['counts']['NORMAL']})")
print(f"test:   {te['n']} (PNEUMONIA {te['counts']['PNEUMONIA']} / "
      f"{te['counts']['PNEUMONIA']/te['n']*100:.1f}%, NORMAL {te['counts']['NORMAL']})")
print("val:     16 (8 / 8) — far too small to tune on; Day 32 builds a")
print("        stratified val split out of train instead.")

train: 5216 (PNEUMONIA 3875 / 74.3%, NORMAL 1341)
test:   624 (PNEUMONIA 390 / 62.5%, NORMAL 234)
val:     16 (8 / 8) — far too small to tune on; Day 32 builds a
        stratified val split out of train instead.


In [2]:
print(f"300-image sample: widths {s['width']['min']}–{s['width']['max']} px, "
      f"heights {s['height']['min']}–{s['height']['max']} px, all {list(s['modes'])[0]}.")
print("Consequence: every image must be resized (224) — scale varies 6x, so the")
print("model must learn scale-invariant features (augmentation helps, Day 32).")
print("Sample grid saved: docs/samples_grid.png (NORMAL top, PNEUMONIA bottom).")

300-image sample: widths 441–2772 px, heights 194–2628 px, all RGB.
Consequence: every image must be resized (224) — scale varies 6x, so the
model must learn scale-invariant features (augmentation helps, Day 32).
Sample grid saved: docs/samples_grid.png (NORMAL top, PNEUMONIA bottom).


## Findings

1. **Imbalance cuts the dangerous way:** always-predict-PNEUMONIA scores 74%
   accuracy on train while missing *every* normal (all false alarms) — and a
   62.5% test accuracy while learning nothing. Accuracy is disqualified on
   day one; precision/recall/F1/ROC-AUC from here on.
2. **Official val (16 images) is unusable** — a single image swings any metric
   by 6 points. Day 32 carves a stratified val split from train.
3. **Geometry varies wildly** (441–2772 px wide) → fixed 224 resize + scale
   augmentation are mandatory, not optional.